# Small CNN 4,996 Parameters MNIST Training

Run these cells top-to-bottom in Google Colab to train the float model, fine-tune with QAT, convert to INT8, then report accuracy, inference time, and model size.


In [ ]:
import copy
import os
import time
from pathlib import Path

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.ao.quantization import (
    DeQuantStub,
    QuantStub,
    convert,
    get_default_qat_qconfig,
    prepare_qat,
)
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")


In [ ]:
# Colab-friendly configuration. Edit these values before retraining.
SEED = 1
DATA_DIR = "./data"
BATCH_SIZE = 64
TEST_BATCH_SIZE = 64
FLOAT_EPOCHS = 14
QAT_EPOCHS = 10
FLOAT_LR = 0.01
QAT_LR = 1e-4
MOMENTUM = 0.9
LOG_INTERVAL = 100
AUGMENT = False
NORMALIZE = False
QCONFIG_BACKEND = "fbgemm"
BENCHMARK_BATCH_SIZE = 1
BENCHMARK_REPEATS = 1000
SAVE_CHECKPOINTS = True
FLOAT_CHECKPOINT = "small_cnn_float_state_dict.pt"
INT8_CHECKPOINT = "small_cnn_int8_state_dict.pt"
INT8_WEIGHTS_TXT = "small_cnn_int8_weights_c_order.txt"
INT8_BIASES_TXT = "small_cnn_biases_float_c_order.txt"
INT8_BIASES_INT32_TXT = "small_cnn_biases_int32_c_order.txt"
INT8_QPARAMS_TXT = "small_cnn_int8_quant_params.txt"
INT8_EXPORT_LAYOUT = "c"  # "c" matches the existing C model layout; "pytorch" keeps module layout.
INFERENCE_SAMPLE_INDEX = 0

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.manual_seed(SEED)
if DEVICE.type == "cuda":
    torch.cuda.manual_seed_all(SEED)

print(f"Using device: {DEVICE}")


In [ ]:
class CNN(nn.Module):
    """Small MNIST CNN with 4,996 trainable parameters."""

    def __init__(self):
        super().__init__()

        self.conv1 = nn.Conv2d(1, 8, kernel_size=3)  # 80 params, output: 8x26x26
        self.conv2 = nn.Conv2d(8, 10, kernel_size=3)  # 730 params, output: 10x11x11
        self.mp = nn.MaxPool2d(2)
        self.fc1 = nn.Linear(10 * 5 * 5, 16)  # 4,016 params
        self.fc2 = nn.Linear(16, 10)  # 170 params

        self.quant = QuantStub()
        self.dequant = DeQuantStub()

    def forward(self, x):
        x = self.quant(x)
        x = self.mp(F.relu(self.conv1(x)))  # 8x13x13
        x = self.mp(F.relu(self.conv2(x)))  # 10x5x5
        x = x.reshape(x.size(0), -1)
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        x = self.dequant(x)
        return F.log_softmax(x, dim=1)


In [ ]:
def build_transform(augment=False, normalize=False):
    transform_steps = []
    if augment:
        transform_steps.append(transforms.RandomAffine(degrees=15, translate=(0.1, 0.1)))
    transform_steps.append(transforms.ToTensor())
    if normalize:
        transform_steps.append(transforms.Normalize((0.1307,), (0.3081,)))
    return transforms.Compose(transform_steps)


def build_loaders(data_dir, batch_size, test_batch_size, augment=False, normalize=False, download=True):
    train_dataset = datasets.MNIST(
        root=str(data_dir),
        train=True,
        transform=build_transform(augment=augment, normalize=normalize),
        download=download,
    )
    test_dataset = datasets.MNIST(
        root=str(data_dir),
        train=False,
        transform=build_transform(augment=False, normalize=normalize),
        download=download,
    )

    loader_kwargs = {
        "num_workers": 2,
        "pin_memory": DEVICE.type == "cuda",
    }
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, **loader_kwargs)
    test_loader = DataLoader(test_dataset, batch_size=test_batch_size, shuffle=False, **loader_kwargs)
    return train_loader, test_loader


train_loader, test_loader = build_loaders(
    data_dir=DATA_DIR,
    batch_size=BATCH_SIZE,
    test_batch_size=TEST_BATCH_SIZE,
    augment=AUGMENT,
    normalize=NORMALIZE,
    download=True,
)

print(f"Train samples: {len(train_loader.dataset)}")
print(f"Test samples: {len(test_loader.dataset)}")


In [ ]:
def count_parameters(model):
    return sum(parameter.numel() for parameter in model.parameters())


def model_size_kb(model, filename="temp.p"):
    path = Path(filename)
    torch.save(model.state_dict(), path)
    try:
        return os.path.getsize(path) / 1024
    finally:
        path.unlink(missing_ok=True)


def sync_if_needed(device):
    if device.type == "cuda":
        torch.cuda.synchronize()


def benchmark_inference(model, device, batch_size=1, warmup=100, repeats=1000):
    model.eval()
    sample = torch.randn(batch_size, 1, 28, 28, device=device)

    with torch.inference_mode():
        for _ in range(warmup):
            model(sample)

        sync_if_needed(device)
        start = time.perf_counter()
        for _ in range(repeats):
            model(sample)
        sync_if_needed(device)
        elapsed_s = time.perf_counter() - start

    images = batch_size * repeats
    ms_per_image = elapsed_s * 1000 / images
    images_per_second = images / elapsed_s
    return ms_per_image, images_per_second


In [ ]:
def train_epoch(model, train_loader, optimizer, epoch, device, log_interval):
    model.train()
    running_loss = 0.0

    for batch_idx, (data, target) in enumerate(train_loader):
        data = data.to(device, non_blocking=True)
        target = target.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)
        output = model(data)
        loss = F.nll_loss(output, target)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * data.size(0)
        if log_interval > 0 and batch_idx % log_interval == 0:
            processed = batch_idx * len(data)
            total = len(train_loader.dataset)
            percent = 100.0 * batch_idx / len(train_loader)
            print(
                f"Train Epoch: {epoch} [{processed}/{total} ({percent:.0f}%)]"
                f" Loss: {loss.item():.6f}"
            )

    return running_loss / len(train_loader.dataset)


def evaluate(model, test_loader, device):
    model.eval()
    test_loss = 0.0
    correct = 0

    with torch.inference_mode():
        for data, target in test_loader:
            data = data.to(device, non_blocking=True)
            target = target.to(device, non_blocking=True)
            output = model(data)

            test_loss += F.nll_loss(output, target, reduction="sum").item()
            pred = output.argmax(dim=1)
            correct += pred.eq(target).sum().item()

    test_loss /= len(test_loader.dataset)
    accuracy = 100.0 * correct / len(test_loader.dataset)
    print()
    print(
        f"Test set: Average loss: {test_loss:.4f}, "
        f"Accuracy: {correct}/{len(test_loader.dataset)} ({accuracy:.2f}%)"
    )
    print()
    return test_loss, correct, accuracy


In [ ]:
def train_float_model(model, train_loader, test_loader, device, epochs, lr, momentum, log_interval):
    optimizer = optim.SGD(model.parameters(), lr=lr, momentum=momentum)
    scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.1)

    for epoch in range(1, epochs + 1):
        train_epoch(model, train_loader, optimizer, epoch, device, log_interval)
        scheduler.step()
        evaluate(model, test_loader, device)

    return model


def train_qat_model(model, train_loader, test_loader, device, epochs, lr, momentum, log_interval, qconfig_backend):
    if qconfig_backend in torch.backends.quantized.supported_engines:
        torch.backends.quantized.engine = qconfig_backend

    qat_model = copy.deepcopy(model)
    qat_model.qconfig = get_default_qat_qconfig(qconfig_backend)
    qat_model.train()
    prepared_model = prepare_qat(qat_model).to(device)
    optimizer = optim.SGD(prepared_model.parameters(), lr=lr, momentum=momentum)

    for epoch in range(1, epochs + 1):
        train_epoch(prepared_model, train_loader, optimizer, epoch, device, log_interval)
        evaluate(prepared_model, test_loader, device)

    return prepared_model


In [ ]:
model = CNN().to(DEVICE)
total_params = count_parameters(model)
print(model)
print(f"Number of parameters: {total_params}")
assert total_params == 4996, f"Expected 4,996 parameters, got {total_params}"


In [ ]:
model = train_float_model(
    model=model,
    train_loader=train_loader,
    test_loader=test_loader,
    device=DEVICE,
    epochs=FLOAT_EPOCHS,
    lr=FLOAT_LR,
    momentum=MOMENTUM,
    log_interval=LOG_INTERVAL,
)


In [ ]:
_float_loss, float_correct, float_accuracy = evaluate(model, test_loader, DEVICE)
float_ms, float_ips = benchmark_inference(
    model,
    device=DEVICE,
    batch_size=BENCHMARK_BATCH_SIZE,
    repeats=BENCHMARK_REPEATS,
)

print(f"Float accuracy: {float_correct}/{len(test_loader.dataset)} ({float_accuracy:.2f}%)")
print(f"Float inference: {float_ms:.6f} ms/image, {float_ips:.2f} images/s")
print(f"Float model size: {model_size_kb(model, 'float.p'):.3f} KB")

if SAVE_CHECKPOINTS:
    torch.save(model.state_dict(), FLOAT_CHECKPOINT)
    print(f"Saved float checkpoint: {FLOAT_CHECKPOINT}")


In [ ]:
prepared_model = train_qat_model(
    model=model,
    train_loader=train_loader,
    test_loader=test_loader,
    device=DEVICE,
    epochs=QAT_EPOCHS,
    lr=QAT_LR,
    momentum=MOMENTUM,
    log_interval=LOG_INTERVAL,
    qconfig_backend=QCONFIG_BACKEND,
)


In [ ]:
_qat_loss, qat_correct, qat_accuracy = evaluate(prepared_model, test_loader, DEVICE)
print(f"QAT accuracy before convert: {qat_correct}/{len(test_loader.dataset)} ({qat_accuracy:.2f}%)")


In [ ]:
model_int8 = convert(prepared_model.to("cpu").eval())
int8_device = torch.device("cpu")
_int8_loss, int8_correct, int8_accuracy = evaluate(model_int8, test_loader, int8_device)
int8_ms, int8_ips = benchmark_inference(
    model_int8,
    device=int8_device,
    batch_size=BENCHMARK_BATCH_SIZE,
    repeats=BENCHMARK_REPEATS,
)

print(f"INT8 accuracy: {int8_correct}/{len(test_loader.dataset)} ({int8_accuracy:.2f}%)")
print(f"INT8 inference: {int8_ms:.6f} ms/image, {int8_ips:.2f} images/s")
print(f"INT8 model size: {model_size_kb(model_int8, 'int8.p'):.3f} KB")

if SAVE_CHECKPOINTS:
    torch.save(model_int8.state_dict(), INT8_CHECKPOINT)
    print(f"Saved INT8 checkpoint: {INT8_CHECKPOINT}")


In [ ]:
def _module_bias_tensor(module):
    bias_attr = getattr(module, "bias", None)
    bias = bias_attr() if callable(bias_attr) else bias_attr
    if bias is None:
        return torch.empty(0, dtype=torch.float32)
    if not torch.is_tensor(bias):
        bias = torch.tensor(bias, dtype=torch.float32)
    return bias.detach().cpu().to(torch.float32).reshape(-1)


def _quantized_weight_tensor(module):
    weight_attr = getattr(module, "weight", None)
    weight = weight_attr() if callable(weight_attr) else weight_attr
    if weight is None or not torch.is_tensor(weight) or not weight.is_quantized:
        raise TypeError(f"{module.__class__.__name__} does not expose a quantized weight tensor")
    return weight.detach().cpu()


def _flatten_int8_weight_for_export(layer_name, qweight, layout="c"):
    int_weight = qweight.int_repr().cpu()
    if layout == "c" and layer_name.startswith("fc"):
        int_weight = int_weight.t().contiguous()
    elif layout not in {"c", "pytorch"}:
        raise ValueError(f"Unsupported export layout: {layout}")
    return int_weight.reshape(-1)


def _weight_qparams(qweight):
    qscheme = qweight.qscheme()
    qscheme_name = str(qscheme)
    if "per_channel" in qscheme_name:
        scales = qweight.q_per_channel_scales().detach().cpu().to(torch.float64).reshape(-1).tolist()
        zero_points = qweight.q_per_channel_zero_points().detach().cpu().to(torch.int64).reshape(-1).tolist()
        axis = int(qweight.q_per_channel_axis())
    else:
        scales = [float(qweight.q_scale())]
        zero_points = [int(qweight.q_zero_point())]
        axis = -1
    return qscheme_name, scales, zero_points, axis


def _expand_per_output(values, output_count):
    if len(values) == output_count:
        return values
    if len(values) == 1:
        return values * output_count
    raise ValueError(f"Expected 1 or {output_count} values, got {len(values)}")


def _activation_qparams_from_quant_stub(quant_stub):
    scale = quant_stub.scale.detach().cpu().reshape(-1)
    zero_point = quant_stub.zero_point.detach().cpu().reshape(-1)
    return float(scale[0].item()), int(zero_point[0].item())


def _activation_qparams_from_module(module):
    return float(module.scale), int(module.zero_point)


def _layer_runtime_specs(model_int8):
    input_scale, input_zero_point = _activation_qparams_from_quant_stub(model_int8.quant)
    conv1_out_scale, conv1_out_zero_point = _activation_qparams_from_module(model_int8.conv1)
    conv2_out_scale, conv2_out_zero_point = _activation_qparams_from_module(model_int8.conv2)
    fc1_out_scale, fc1_out_zero_point = _activation_qparams_from_module(model_int8.fc1)
    fc2_out_scale, fc2_out_zero_point = _activation_qparams_from_module(model_int8.fc2)
    return [
        {
            "name": "conv1",
            "module": model_int8.conv1,
            "input_scale": input_scale,
            "input_zero_point": input_zero_point,
            "output_scale": conv1_out_scale,
            "output_zero_point": conv1_out_zero_point,
        },
        {
            "name": "conv2",
            "module": model_int8.conv2,
            "input_scale": conv1_out_scale,
            "input_zero_point": conv1_out_zero_point,
            "output_scale": conv2_out_scale,
            "output_zero_point": conv2_out_zero_point,
        },
        {
            "name": "fc1",
            "module": model_int8.fc1,
            "input_scale": conv2_out_scale,
            "input_zero_point": conv2_out_zero_point,
            "output_scale": fc1_out_scale,
            "output_zero_point": fc1_out_zero_point,
        },
        {
            "name": "fc2",
            "module": model_int8.fc2,
            "input_scale": fc1_out_scale,
            "input_zero_point": fc1_out_zero_point,
            "output_scale": fc2_out_scale,
            "output_zero_point": fc2_out_zero_point,
        },
    ]


def export_int8_weights_and_metadata(
    model_int8,
    weights_path=INT8_WEIGHTS_TXT,
    biases_path=INT8_BIASES_TXT,
    biases_int32_path=INT8_BIASES_INT32_TXT,
    qparams_path=INT8_QPARAMS_TXT,
    layout=INT8_EXPORT_LAYOUT,
):
    runtime_specs = _layer_runtime_specs(model_int8)
    all_weights = []
    all_biases_float = []
    all_biases_int32 = []
    qparam_lines = [
        "# Quantized INT8 export for small_cnn_4996_params",
        f"layout={layout}",
        f"model_input_scale={runtime_specs[0]['input_scale']:.10g}",
        f"model_input_zero_point={runtime_specs[0]['input_zero_point']}",
        "# Weight file contains signed INT8 values, one scalar per line.",
        "# Float bias file contains one scalar per line.",
        "# INT32 bias file uses round(bias_float / (input_scale * weight_scale)).",
        "# conv2 input qparams are conv1 output qparams after maxpool.",
        "# fc1 input qparams are conv2 output qparams after maxpool and reshape.",
        "# fc2 input qparams are fc1 output qparams after relu.",
        "",
    ]

    for spec in runtime_specs:
        layer_name = spec["name"]
        module = spec["module"]
        qweight = _quantized_weight_tensor(module)
        flat_weight = _flatten_int8_weight_for_export(layer_name, qweight, layout=layout)
        bias_float = _module_bias_tensor(module)
        qscheme_name, weight_scales, weight_zero_points, axis = _weight_qparams(qweight)
        weight_scales = _expand_per_output(weight_scales, bias_float.numel())
        weight_zero_points = _expand_per_output(weight_zero_points, bias_float.numel())

        bias_scales = [spec["input_scale"] * weight_scale for weight_scale in weight_scales]
        bias_int32 = [
            int(round(float(bias_value) / bias_scale)) if bias_scale != 0 else 0
            for bias_value, bias_scale in zip(bias_float.tolist(), bias_scales)
        ]
        requant_scales = [bias_scale / spec["output_scale"] for bias_scale in bias_scales]

        all_weights.extend(int(value) for value in flat_weight.tolist())
        all_biases_float.extend(float(value) for value in bias_float.tolist())
        all_biases_int32.extend(bias_int32)

        qparam_lines.extend([
            f"[{layer_name}]",
            f"weight_shape={list(qweight.shape)}",
            f"exported_weight_count={flat_weight.numel()}",
            f"bias_count={bias_float.numel()}",
            f"input_scale={spec['input_scale']:.10g}",
            f"input_zero_point={spec['input_zero_point']}",
            f"output_scale={spec['output_scale']:.10g}",
            f"output_zero_point={spec['output_zero_point']}",
            f"weight_qscheme={qscheme_name}",
            f"weight_per_channel_axis={axis}",
            "weight_scales=" + " ".join(f"{value:.10g}" for value in weight_scales),
            "weight_zero_points=" + " ".join(str(int(value)) for value in weight_zero_points),
            "bias_scales=" + " ".join(f"{value:.12g}" for value in bias_scales),
            "requant_scales=" + " ".join(f"{value:.12g}" for value in requant_scales),
            "",
        ])

    weights_path = Path(weights_path)
    biases_path = Path(biases_path)
    biases_int32_path = Path(biases_int32_path)
    qparams_path = Path(qparams_path)
    for output_path in (weights_path, biases_path, biases_int32_path, qparams_path):
        output_path.parent.mkdir(parents=True, exist_ok=True)

    weights_path.write_text("".join(f"{value}\n" for value in all_weights))
    biases_path.write_text("".join(f"{value:.10g}\n" for value in all_biases_float))
    biases_int32_path.write_text("".join(f"{value}\n" for value in all_biases_int32))
    qparams_path.write_text("\n".join(qparam_lines) + "\n")

    export_info = {
        "weights_path": str(weights_path),
        "biases_path": str(biases_path),
        "biases_int32_path": str(biases_int32_path),
        "qparams_path": str(qparams_path),
        "weight_count": len(all_weights),
        "bias_count": len(all_biases_float),
        "bias_int32_count": len(all_biases_int32),
    }
    print(f"Exported {len(all_weights)} signed INT8 weight values to {weights_path}")
    print(f"Exported {len(all_biases_float)} float bias values to {biases_path}")
    print(f"Exported {len(all_biases_int32)} INT32 bias values to {biases_int32_path}")
    print(f"Exported activation and weight quantization parameters to {qparams_path}")
    return export_info


int8_export_info = export_int8_weights_and_metadata(model_int8)


In [ ]:
def run_single_inference(model, dataset, sample_index=0, device=torch.device("cpu")):
    model.eval()
    image, label = dataset[sample_index]
    image = image.unsqueeze(0).to(device)

    with torch.inference_mode():
        output = model(image)
        probabilities = output.exp()
        confidence, prediction = probabilities.max(dim=1)

    result = {
        "sample_index": sample_index,
        "prediction": int(prediction.item()),
        "label": int(label),
        "confidence": float(confidence.item()),
        "correct": int(prediction.item()) == int(label),
        "log_probabilities": output.squeeze(0).detach().cpu().tolist(),
    }
    print(f"Sample index: {result['sample_index']}")
    print(f"Prediction: {result['prediction']}")
    print(f"Label: {result['label']}")
    print(f"Confidence: {result['confidence']:.4f}")
    print(f"Correct: {result['correct']}")
    return result


int8_inference_result = run_single_inference(
    model_int8,
    test_loader.dataset,
    sample_index=INFERENCE_SAMPLE_INDEX,
    device=torch.device("cpu"),
)


In [ ]:
print("Summary")
print(f"Float accuracy: {float_accuracy:.2f}%")
print(f"QAT accuracy before convert: {qat_accuracy:.2f}%")
print(f"INT8 accuracy: {int8_accuracy:.2f}%")
print(f"Float inference: {float_ms:.6f} ms/image")
print(f"INT8 inference: {int8_ms:.6f} ms/image")
if "int8_export_info" in globals():
    print(f"INT8 weights: {int8_export_info['weights_path']} ({int8_export_info['weight_count']} values)")
    print(f"INT8 biases: {int8_export_info['biases_path']} ({int8_export_info['bias_count']} values)")
    print(f"INT8 bias_int32: {int8_export_info['biases_int32_path']} ({int8_export_info['bias_int32_count']} values)")
    print(f"INT8 qparams: {int8_export_info['qparams_path']}")
if "int8_inference_result" in globals():
    print(
        f"INT8 sample {int8_inference_result['sample_index']}: "
        f"pred={int8_inference_result['prediction']}, "
        f"label={int8_inference_result['label']}, "
        f"correct={int8_inference_result['correct']}"
    )
